# Comparison of urban indicators in Tampere and Helsinki, Finland

The following Jupyter notebook calculates some urban indicators for Tampere and Helsinki.  

In [1]:
import pathlib 
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"
OUTPUT_DIRECTORY = NOTEBOOK_PATH / "docs"

In [2]:
# Most necessary imports
import json
import os
import geopandas as gpd
import pandas as pd
# import requests
# from datetime import datetime
import matplotlib.pyplot as plt
# import contextily as cx
import networkx as nx
import osmnx as ox

#import street_network_metrics as net

from functools import partial
from geopy.geocoders import Nominatim

os.environ['PROJ_LIB'] = '/path/to/env/share/proj'
#os.environ['GDAL_DATA'] = '/path/to/env/share/'
pd.set_option("display.max_rows", None)

In [3]:
CITIES = CITY1, CITY2 = ('Helsinki, Finland', 'Tampere, Finland')

In [4]:
#ox.settings.overpass_endpoint = 'https://overpass.kumi.systems/api/interpreter'

In [5]:
def get_network(osm_city):
    assert type(osm_city) == str, 'You should provide a string'
    assert osm_city is not None, 'You should provide a city name'
    graph = ox.graph_from_place(osm_city, network_type='all')
    print(type(graph))
    assert graph is not None, 'No graphs returned by the query'
    return osm_city, graph

In [6]:
# nets_t = get_networks([])
# for G in nets:
#     print(f)
nets = { c: {'graph': g} for c, g in [get_network(city) for city in CITIES]}

C:\Users\eemeli.pettersson\Anaconda3\envs\geopython\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))


<class 'networkx.classes.multidigraph.MultiDiGraph'>


C:\Users\eemeli.pettersson\Anaconda3\envs\geopython\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))


<class 'networkx.classes.multidigraph.MultiDiGraph'>


In [7]:
print(type(nets))
print(len(nets.keys()))
for key in nets:
    print(nets[key])
    #ox.plot_graph(G)

<class 'dict'>
2
{'graph': <networkx.classes.multidigraph.MultiDiGraph object at 0x00000219B0901B20>}
{'graph': <networkx.classes.multidigraph.MultiDiGraph object at 0x000002197CC9B9D0>}


In [8]:
def graphs_to_gdfs(graph_objs):
    assert len(graph_objs.keys()) > 0, 'No graphs to convert'
    for key in graph_objs:
        graph = ox.project_graph(graph_objs[key]['graph'])
        nodes, edges = ox.graph_to_gdfs(graph)
        graph_objs[key]['edges'] = edges
        graph_objs[key]['nodes'] = nodes
    return graph_objs

In [9]:
gdf_dict = graphs_to_gdfs(nets)


In [46]:
def gdf_exploded_counts(cityname, gdf, col_name, type_title):
    gdf_exp = gdf.explode(col_name)
    renamed_col = 'Count ({city})'.format(city=cityname)
    counts = pd.DataFrame(gdf_exp[col_name].value_counts().reset_index())
    counts[col_name] = counts[col_name].astype(pd.Int64Dtype())
    counts = counts.rename({col_name: renamed_col, 'index': type_title}, axis=1)
    return renamed_col, counts

In [47]:
def join_highway_type_counts(gdf_dict, join_col):
    for city in gdf_dict:
        gdf = gdf_dict[city]['edges']
        count_col, counts = gdf_exploded_counts(city, gdf, 'highway', join_col)
        gdf_dict[city]['counts'] = counts
    left, right = gdf_dict[CITY1]['counts'], gdf_dict[CITY2]['counts']
    joined_counts = left.merge(right=right, left_on=join_col, right_on=join_col, how='outer')
    return joined_counts

In [61]:
total_counts = join_highway_type_counts(gdf_dict, 'Roadlink types')
total_counts = total_counts.set_index('Roadlink types')
total_counts.replace({pd.NA: 0})
print(total_counts.info())

<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, footway to bus_stop
Data columns (total 2 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Count (Helsinki, Finland)  23 non-null     Int64
 1   Count (Tampere, Finland)   24 non-null     Int64
dtypes: Int64(2)
memory usage: 650.0+ bytes
None


In [63]:
total_counts_styled = total_counts.style.set_table_attributes("style='display:inline'").set_caption(
    'Number of roadlinks per roadlink type in {city1} and {city2}. Some roadlinks belong to several categories.'.format(city1=CITY1, city2=CITY2)
)
total_counts_styled

,"Count (Helsinki, Finland)","Count (Tampere, Finland)"
Roadlink types,,
footway,168943,20655
service,74551,26346
cycleway,67654,27802
residential,41934,20469
path,39726,27432
steps,10710,880
tertiary,9982,4637
unclassified,7049,3140
secondary,6873,2000


In [ ]:
# helsinki = pd.DataFrame(gdf_dict[CITIES[0]]['edges']['highway'].value_counts().reset_index())
# tampere = pd.DataFrame(gdf_dict[CITIES[1]]['edges']['highway'].value_counts().reset_index())

cities_edges = [pd.DataFrame(gdf_dict[city]['edges']['highway'].value_counts().reset_index()) for city in CITIES]
new_dfs = []

def list_to_str(ls):
    #print(type(ls))
    #print(ls)
    if all([len(elem) > 1 and type(elem) == str for elem in ls]):
        return ", ".join(ls)
    else:
        return ls

for i in range(len(cities_edges)):
    df = cities_edges[i]
    df['Way types'] = df['index'].apply(list_to_str)
    highway_new = 'Count ({city})'.format(city=CITIES[i])
    df = df.rename({'highway': highway_new}, axis=1)[['Way types', highway_new]]
    new_dfs.append(df)
joined = new_dfs[0].merge(right=new_dfs[1], left_on='Way types', right_on='Way types', how='outer').drop(axis=1, )
joined.style.set_table_attributes("style='display:inline'").set_caption('Table name}')
joined = joined.set_index(keys=joined['Way types'], drop=True)
print(joined.head(1000))
#type(res)
# for df, city in zip(cities_edges, CITIES):
#     df.style.set_table_attributes("style='display:inline'").set_caption('Table name}')
#     print(df)

ValueError: Need to specify at least one of 'labels', 'index' or 'columns'

In [ ]:
# def graph_dict_to_strings(gdf_dict, dir_path=None):
#     strings = {}
#     for key in sorted(gdf_dict.keys()):
#         string = ''
#         string += 'City: {key}\n'.format(key=key)
#         string += 'CRS EPSG: {crs}\n'.format(crs=gdf_dict[key]['edges'].crs.to_epsg())
#         string += 'Index: {index}\n'.format(index=gdf_dict[key]['edges'].index.names)
#         string += 'Value counts\n:{val_counts}\n'.format(val_counts=gdf_dict[key]['edges']['highway'].value_counts())
#         strings[key] = string
#     # if dir_path is not None:
#     #     assert type(dir_path) != str or type(dir_path) != pathlib.Path 
#     #     assert type(path) == pathlib.Path
#     #     with open(dir_path / 
#     return strings
        
        

In [ ]:
graph_strings = graph_dict_to_strings(gdf_dict)
for key in sorted(graph_strings.keys()):
    print(graph_strings[key])
    print('\n')

City: Helsinki, Finland
CRS EPSG: 32635
Index: ['u', 'v', 'key']
Value counts
:footway                                   159415
service                                    70375
cycleway                                   66290
residential                                40416
path                                       38122
                                           ...  
[path, residential, footway]                   1
[service, unclassified, living_street]         1
[primary, secondary]                           1
[trunk, motorway]                              1
[living_street, unclassified, service]         1
Name: highway, Length: 129, dtype: int64



City: Tampere, Finland
CRS EPSG: 32634
Index: ['u', 'v', 'key']
Value counts
:cycleway                              26924
path                                  26600
service                               25124
footway                               19467
residential                           19259
                                      ..